# Modèle Numérique de Surface (MNS) et Modèle Numérique de Hauteur (MNH)

Ce notebook vérifie `processing.lidar.rasterize.compute_mns` (rasterisation de l'ensemble du nuage, valeur maximale par cellule = sursol) et `compute_mnh` (hauteur au-dessus du sol via `filters.hag_nn`, sans avoir à produire ni aligner deux rasters séparés), sur le nuage fusionné produit par le notebook 15.

La cohérence est vérifiée en comparant, pixel par pixel, `MNH` avec `MNS - MNT` (notebook 18).

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

import numpy as np
import rasterio

from processing.lidar.rasterize import compute_mns, compute_mnh, compute_mnt

merged_path = repo_root / "data" / "raw" / "raster" / "lidar" / "nuage_points.las"
mnt_path = repo_root / "data" / "processed" / "raster" / "lidar" / "mnt.tif"
mns_path = repo_root / "data" / "processed" / "raster" / "lidar" / "mns.tif"
mnh_path = repo_root / "data" / "processed" / "raster" / "lidar" / "mnh.tif"

## 1. Génération du MNS

In [ ]:
result_mns = compute_mns(merged_path, mns_path, resolution=0.5)

with rasterio.open(result_mns) as src:
    print("CRS :", src.crs)
    print("Resolution :", src.res)
    print("Bounds :", src.bounds)
    mns = src.read(1)

valid_mns = mns[mns != -9999]
print(f"\nPixels valides : {valid_mns.size} / {mns.size}")
print(f"Altitude MNS min/max/moyenne : {valid_mns.min():.2f} / {valid_mns.max():.2f} / {valid_mns.mean():.2f} m")

## 2. Génération du MNH (hauteur au-dessus du sol, `filters.hag_nn`)

In [ ]:
result_mnh = compute_mnh(merged_path, mnh_path, resolution=0.5)

with rasterio.open(result_mnh) as src:
    print("CRS :", src.crs)
    print("Resolution :", src.res)
    print("Bounds :", src.bounds)
    mnh = src.read(1)

valid_mnh = mnh[mnh != -9999]
print(f"\nPixels valides : {valid_mnh.size} / {mnh.size}")
print(f"Hauteur MNH min/max/moyenne : {valid_mnh.min():.2f} / {valid_mnh.max():.2f} / {valid_mnh.mean():.2f} m")

## 3. Cohérence : MNH vs MNS − MNT

In [ ]:
result_mnt = compute_mnt(merged_path, mnt_path, resolution=0.5)

with rasterio.open(result_mnt) as src:
    mnt = src.read(1)

mask = (mnt != -9999) & (mns != -9999) & (mnh != -9999)
diff = (mns[mask] - mnt[mask]) - mnh[mask]

print(f"Pixels compares : {mask.sum()}")
print(f"Ecart (MNS-MNT) vs MNH : moyenne={diff.mean():.3f} m, ecart-type={diff.std():.3f} m, max abs={np.abs(diff).max():.3f} m")
print("\n(Un ecart faible est attendu : le MNT n'est defini que la ou il y a des points sol,")
print("alors que le hag_nn interpole le sol sous chaque point, y compris sous le bati/la vegetation.)")

## 4. Aperçu du MNS et du MNH

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(result_mns) as s:
    extent = [s.bounds.left, s.bounds.right, s.bounds.bottom, s.bounds.top]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

im0 = axes[0].imshow(np.ma.masked_equal(mns, -9999), cmap="terrain", extent=extent)
axes[0].set_title("MNS (altitude sursol, m)")
fig.colorbar(im0, ax=axes[0], label="Altitude (m)", fraction=0.046)

im1 = axes[1].imshow(np.ma.masked_equal(mnh, -9999), cmap="viridis", vmin=0, vmax=30, extent=extent)
axes[1].set_title("MNH (hauteur au-dessus du sol, m)")
fig.colorbar(im1, ax=axes[1], label="Hauteur (m)", fraction=0.046)

plt.show()